# Arquitectura Cortex R y A

Presentación: file://wsl.localhost/Ubuntu/home/mariano/marianoTD2/Arquitectura%20Cortex%20R%20%20A.pdf


<img src="./versiones.png" width="1000" />


## Cortex-R

- Cortex-R4: Pensado para aplicaciones de embebidos en **entornos de seguridad**. Puede ejecutar instrucciones en modo **ARM** (set de instrucciones de 32 bits) y **THUMB-2** (Cortex-M de 16 bits). Gestor de interrupciones GIC. LLegan hasta los **600MHz** de frecuencia típica de clock.

- Cortex-R5: Similar al R4 pero puede utilizar **multicore de dos procesadores** (**AMP**) (multinúcleo asimétrico: en un mismo encapsulado, hay varios procesadores que realizan tareas de forma independiente) (multicore simétrico: popular en celulares, los procesadores realizan una misma tarea. No estoy agregando capacidad de procesamiento, pero separo una misma tarea en varios cores para poder hacerlo mas eficiente).

- Cortex-R7: La rama mas grande de Cortex-R Trabaja en **ejecución fuera de orden** con **pipeline de 11 etapas** y pudiendo
trabajar con un **reloj** de **1GHz** típicamente. Soporta **división por hardware**, **DSP** (procesamiento de señales) y **punto flotante**. También soporta **instrucciones en modo ARM y modo Thumb-2**.



<img src="./cortexr4.png" width="1000" />

Las memorias representan la mayor parte del costo de estos micros, esto debido a la baja latencia de las mismas

### Modos de operación

Son 7 modos de operación de los cuales el único modo **sin privilegios** es el modo **usuario** (no tenemos acceso al registro de control para modificar el comportamiento del sistema).

- Modo sistema: Acá corre el sistema operativo.

- Modo IRQ y FIQ: Acá corren las interrupciones.

- Modo Abort: Se activa en caso de haber una violación de memoria.

- Modo Undef: Se activa para manejar instrucciones no definidas.

- Modo Supervisor (SVC): Se activa al entrar en reset luego de una interrupción.

Estos modos tienen registros asociados:

<img src="./registros cortex r.png" width="1000" />

Para los modos FIQ e IRQ tenemos set de instrucciones ya almacenados en los registros en celeste para atender interrupciones de manera mas rápida. Hay 6 Stack Points. Cada uno habilita registros especiales de estado (CPSR y SPSR) (por ejemplo, que interrupción se ejecuto y estado del programa). Esto lo maneja un sistema operativo. 
### CPSR
Contiene los flags de operación (ya lo ví con Furfi pero lo tengo que repasar)

<img src="./CPSR.png" width="1000" />

### Pipeline:

<img src="./Pipeline.png" width="1000" />

### Unidad de punto Flotante

Presición **simple** (32 bits) y **doble** (64 bits)

**Punto fijo vs Punto flotante:** La diferencia principal radica en cómo ubican el punto decimal. El punto fijo utiliza una cantidad fija de dígitos para la parte entera y para la fracción. El punto flotante permite que el punto decimal "flote", almacenando el número en formato de notación científica (signo, mantisa y exponente) para representar un rango mucho más amplio.

Las unidades de punto flotante pueden realizar conversiones entre ambos sistemas.

<img src="./VFP.png" width="1000" />


### Flags

Se usan para establecer proposiciones y lineas condicionales para efectuar bifurcaciones en un programa

### Tabla de Excepciones (el maldito TP0 de Furfaro)

Se define una lista de exepciones donde deben estar los handlers para cada tipo de excepción en caso de activar por ejemplo el modo FIQ o IRQ, en este caso se deshabilitan automáticamente.

<img src="./excepciones.png" width="1000" />


### Manejo de Interrupciones

<img src="./manejo de interrupciones.png" width="1000" />

- Modo indexado: El periferico pide la **IRQ/FIQ** al controlador de interrupciones. Se genera la **IRQ o FIQ**. Luego, la cpu salta a la dirección **0x18 o 0x1C** y se carga una **ISR**

- Modo vectorizado por registro: En la dirección **0x18 o 0x1C** se encuentra la instrucción

  Ejemplo:
    LDR PC, [PC, #-0x1B0]  que apunta al registro
    que tiene la dirección de la ISR particular.

- Modo vectorizado por hardware: El controlador de interrupciones tiene acceso a las posiciones de memoria 0x18 y 0x1C y directamente coloca ahí la dirección de la ISR particular

### Modo Abort

<img src="./modo abort.png" width="1000" />

### Cache y TCM

El cache externo para manejar el bus de datos y en el interno se encuentra código de uso frecuente.

La principal diferencia entre TCM y Cache es que la primera forma parte del mapa de memoria, y por tanto, yo tengo acceso a la misma.

### Inicio de un Cortex-R

<img src="./inicio.png" width="1000" />

### Modos de Bajo consumo

<img src="./bajo consumo.png" width="1000" />

En el modo StandBy es posible que haga falta reiniciar algún módulo del procesador

## Cortex-A

<img src="./cortexa.png" width="1000" />

<img src="./proca.png" width="1000" />

### Modos de ejecución

Son exactamente iguales a los Cortex-R

### Zonas seguras

El usuario puede proteger datos y código para ser accesible solo a través de modos privilegiados.

**Niveles de privilegio:** Modo Hypervisor y modo Virtualización.

- Hypervisor:

- Virtualización:

<img src="./modos de privilegio.png" width="1000" />

### Registros


<img src="./registros a.png" width="1000" />


### Instrucciones SIMD

<img src="./SIMD.png" width="1000" />

Las interrupciones convienen ejecutarlas en modo seguro, esto debido a que se destinan a atender periféricos, y si estos son de comunicación son vulnerables a ataques.

### Multiprocesamiento

<img src="./multiproc.png" width="1000" />

# Parte práctica

Algoritmo de pulsador:

- Preguntar nivel

- Comparar con el nivel anterior
{
      	si ==

          incremento cuenta

              si cuenta >= Threshold

                  valido nivel

      si !=

          cuenta = 0

}

El umbral va a ser un define